# Google Colab Setup for MK-UNet Coconut Plantation Training

This notebook demonstrates how to set up and run the MK-UNet training pipeline on Google Colab with access to your Google Drive data.

## 1. Import Required Libraries

Import necessary libraries including google.colab for authentication and file system operations.

In [ ]:
# Import Required Libraries
import os
import sys

# Google Colab specific imports
try:
    from google.colab import drive, auth
    IN_COLAB = True
    print("✓ Running in Google Colab environment")
except ImportError:
    IN_COLAB = False
    print("✗ Not running in Google Colab")

# Standard imports
import numpy as np
import torch
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Authenticate with Google Account

Use the google.colab.auth module to authenticate your Google account and obtain credentials for accessing Google services.

In [ ]:
# Authenticate with Google Account
if IN_COLAB:
    print("Starting Google authentication...")
    auth.authenticate_user()
    print("✓ Successfully authenticated!")
else:
    print("⚠ Skipping authentication (not in Colab)")

## 3. Mount Google Drive

Mount your Google Drive to the Colab environment to access your project files and datasets.

In [ ]:
# Mount Google Drive
if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)
    print("✓ Google Drive mounted successfully!")
    
    # Change to the project directory
    # Update this path based on where you uploaded your project in Google Drive
    project_path = '/content/drive/MyDrive/MK-UNet-main'  # Modify this path as needed
    
    if os.path.exists(project_path):
        os.chdir(project_path)
        print(f"✓ Changed directory to: {os.getcwd()}")
    else:
        print(f"⚠ Project path not found: {project_path}")
        print("  Available directories in Google Drive:")
        drive_path = '/content/drive/MyDrive'
        if os.path.exists(drive_path):
            for item in os.listdir(drive_path)[:10]:  # Show first 10 items
                print(f"    - {item}")
else:
    print("⚠ Not in Colab, skipping drive mount")

## 4. Verify Connection and Access Files

Verify the successful connection by listing project files and checking dataset accessibility.

In [ ]:
# Verify Connection and List Project Files
print("Current working directory:", os.getcwd())
print("\n📁 Project files:")
if os.path.exists('.'):
    for item in sorted(os.listdir('.'))[:15]:  # Show first 15 items
        path = os.path.join('.', item)
        item_type = '📁' if os.path.isdir(path) else '📄'
        print(f"  {item_type} {item}")

# Check if key components exist
print("\n🔍 Checking key components:")
components = [
    'mkunet_network.py',
    'train_polyp.py',
    'utils/dataloader_polyp.py',
    'data/polyp/target/ClinicDB/train/images',
    'annotations'
]

for component in components:
    exists = "✓" if os.path.exists(component) else "✗"
    print(f"  {exists} {component}")

## 5. Install Dependencies

Install required Python packages for the MK-UNet project.

In [ ]:
# Install Required Packages
import subprocess

# Read requirements.txt if it exists
requirements = []
if os.path.exists('requirements.txt'):
    with open('requirements.txt', 'r') as f:
        requirements = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    print("📦 Installing requirements from requirements.txt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + requirements)
    print("✓ Dependencies installed successfully!")
else:
    print("⚠ requirements.txt not found")

# Upgrade pip and install specific packages if needed
print("\n📦 Installing/Upgrading core packages...")
core_packages = ['torch', 'torchvision', 'numpy', 'scipy', 'scikit-image', 'opencv-python', 'PIL']
# pip install will skip if already installed with compatible version
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

## 6. Check GPU/TPU Resources

Verify available computational resources in the Colab environment.

In [ ]:
# Check GPU and TPU Resources
print("🖥️  Computing Resources:")
print(f"  GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Count: {torch.cuda.device_count()}")
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Allocate a small tensor to verify GPU works
    test_tensor = torch.randn(10, 10).cuda()
    print(f"  ✓ GPU is working!")
else:
    print("  ⚠ No GPU available - CPU will be used (training will be slow)")

# Check RAM
import psutil
total_ram = psutil.virtual_memory().total / 1e9
available_ram = psutil.virtual_memory().available / 1e9
print(f"\n  RAM Available: {available_ram:.2f} GB / {total_ram:.2f} GB")

## 7. Execute Training Code

Run the MK-UNet training pipeline with your customized parameters.

In [ ]:
# Configure Training Parameters
import argparse

# Create training configuration
training_config = {
    'network': 'MK_UNet',  # Options: 'MK_UNet_T', 'MK_UNet_S', 'MK_UNet', 'MK_UNet_M', 'MK_UNet_L'
    'epoch': 50,  # Reduced from 200 for faster testing
    'lr': 0.0005,
    'batchsize': 4,  # Reduced for Colab GPU memory - adjust based on your GPU
    'test_batchsize': 4,
    'img_size': 352,
    'clip': 0.5,
    'decay_rate': 0.1,
    'decay_epoch': 300,
    'color_image': True,
    'augmentation': True,
    'train_path': './data/polyp/target/ClinicDB/train/',
    'test_path': './data/polyp/target/ClinicDB/',
}

print("🚀 Training Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

In [ ]:
# Run Training Script
print("Starting training...\n")

# Build the command
cmd = [
    sys.executable, 'train_polyp.py',
    '--network', training_config['network'],
    '--epoch', str(training_config['epoch']),
    '--lr', str(training_config['lr']),
    '--batchsize', str(training_config['batchsize']),
    '--test_batchsize', str(training_config['test_batchsize']),
    '--img_size', str(training_config['img_size']),
    '--clip', str(training_config['clip']),
    '--decay_rate', str(training_config['decay_rate']),
    '--decay_epoch', str(training_config['decay_epoch']),
    '--color_image', str(training_config['color_image']),
    '--augmentation', str(training_config['augmentation']),
]

# Execute the training
try:
    subprocess.run(cmd, check=True)
    print("\n✓ Training completed successfully!")
except subprocess.CalledProcessError as e:
    print(f"\n✗ Training failed with error: {e}")

## 8. Post-Training Operations

Save and download your trained models and results.

In [ ]:
# List and organize results
print("📊 Training Results Summary:\n")

# Check model outputs
if os.path.exists('model_pth'):
    print("✓ Trained models saved in: ./model_pth/")
    for folder in os.listdir('model_pth'):
        model_folder = os.path.join('model_pth', folder)
        if os.path.isdir(model_folder):
            files = os.listdir(model_folder)
            print(f"\n  📁 {folder}")
            for file in files:
                print(f"    - {file}")

# Check logs
if os.path.exists('logs'):
    print("\n✓ Training logs saved in: ./logs/")
    for log_file in sorted(os.listdir('logs'))[:5]:  # Show first 5
        print(f"  - {log_file}")

print("\n💾 To save your results to Google Drive:")
print("  Results are automatically synced since the project is in Google Drive!")
print("  You can access them at: /content/drive/MyDrive/MK-UNet-main/model_pth/")

## Quick Start Guide

### 📋 Step-by-Step Instructions:

1. **Prepare Your Project in Google Drive:**
   - Upload your MK-UNet-main folder to Google Drive
   - Ensure all data, annotations, and scripts are included

2. **Upload This Notebook:**
   - Save this notebook to your Google Drive
   - Open it in Google Colab (Right-click → Open with → Google Colaboratory)

3. **Run Cells Sequentially:**
   - Execute each cell from top to bottom
   - The first authentication cell will prompt you to authorize access
   - Update the `project_path` in cell 3 if needed (check available directories)

4. **Customize Training:**
   - Modify `training_config` in cell 7 based on your GPU capacity
   - Reduce `batchsize` if you get out of memory errors
   - Adjust `epoch` for faster testing

5. **Monitor Training:**
   - Watch the console output for training progress
   - Training logs are saved in `logs/` directory
   - Models are saved in `model_pth/` directory

### ⚙️ Useful Tips:

- **Memory Issues:** Reduce `batchsize` or `img_size`
- **Slow Training:** Consider using a smaller network variant (e.g., MK_UNet_S)
- **Session Timeout:** Colab notebooks run for up to 12 hours; save frequently to Drive
- **Check GPU:** Run the GPU check cell before each training session

### 🔗 Resources:

- [Google Colab Documentation](https://colab.research.google.com/)
- [PyTorch Documentation](https://pytorch.org/docs/)